In [ ]:
import cv2
import numpy as np
from skimage.morphology import skeletonize
import matplotlib.pyplot as plt

def apply_adaptive_threshold(blurred_gray_image):
    """
    Applies Gaussian Adaptive Thresholding to handle uneven lighting.
    """
    # Parameters to tune based on your specific cloud chamber:
    # blockSize: Size of a pixel neighborhood (must be an odd number like 11, 21, 31).
    #            Larger blocks handle thicker tracks better.
    # C: Constant subtracted from the mean. Higher C = stricter noise filtering.
    
    block_size = 15 
    c_value = 15    

    # Apply adaptive thresholding
    binary = cv2.adaptiveThreshold(
        blurred_gray_image,
        255,                                  # Maximum value (White)
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,       # Calculates weighted sum of neighborhood
        cv2.THRESH_BINARY,
        block_size,
        c_value
    )
    
    # Cloud chamber backgrounds are dark and tracks are light.
    # If the thresholding inverts this, uncomment the line below:
    # binary = cv2.bitwise_not(binary)
    
    return binary

def process_track_to_skeleton(image_path):
    """
    Takes an image of a particle track and returns its 1-pixel-wide skeleton.
    """
    # 1. Load the image in grayscale
    # Cloud chamber tracks only need intensity data, not color.
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not load image at {image_path}")

    # 2. Pre-processing: Blur
    # Wispy tracks (like electrons) have noisy edges. A slight Gaussian blur 
    # smooths the track so it doesn't generate "false branches" later.
    blurred_temp = cv2.GaussianBlur(img, (5, 5), 0)
    blurred_temp2 = cv2.GaussianBlur(img, (5, 5), 0)
    blurred = cv2.GaussianBlur(img, (23, 23), 0)

    clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8,8)
    )

    

    blur2 = cv2.fastNlMeansDenoising(img, None, h=40, templateWindowSize=7, searchWindowSize=21)

    enhanced = clahe.apply(blur2)
    
    # 3. Binarization: Adaptive Thresholding
    # Cloud chamber lighting varies. Adaptive thresholding handles uneven lighting better.
    binary = apply_adaptive_threshold(blur2)

    # 4. Gap Bridging: Morphological Closing
    # Sometimes a track has a 1-pixel gap. Closing dilates the track to bridge gaps,
    # then erodes it back to its original size.
    kernel = np.ones((3, 3), np.uint8)
    
    closed_binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    
    # 5. Skeletonization
    # skimage requires a boolean array (True/False), not 0-255 pixel values.
    boolean_mask = closed_binary > 0
    skeleton = skeletonize(boolean_mask)

    return img, blurred, enhanced, blur2, closed_binary, skeleton

# ==========================================
# Visualization & Testing
# ==========================================
if __name__ == "__main__":
    # Replace with your actual annotated image path
    path = "test.jpg"
    
    try:
        original, blurred, enhanced, blur2, binarized, skeleton = process_track_to_skeleton(path)
        
        # Display the transformation steps side-by-side
        fig, axes = plt.subplots(1, 6, figsize=(15, 5))
        
        axes[0].imshow(original, cmap='gray')
        axes[0].set_title('1. Original Cropped Track')
        
        axes[1].imshow(blurred, cmap='gray')
        axes[1].set_title('2. Blurred Image')
        
        axes[2].imshow(enhanced, cmap='gray')
        axes[2].set_title('3. CLAHE')
        
        axes[3].imshow(blur2, cmap='gray')
        axes[3].set_title('4. Fast Means Denoising')
        
        axes[4].imshow(binarized, cmap='gray')
        axes[4].set_title('5. Cleaned Binary Mask')
        
        axes[5].imshow(skeleton, cmap='gray')
        axes[5].set_title('6. 1-Pixel Skeleton')
        
        for ax in axes:
            ax.axis('off')
            
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(e)

ModuleNotFoundError: No module named 'cv2'

In [ ]:
python pip install opencv-python

pip install matplotlib

pip install scikit-image

pip install numpy

SyntaxError: invalid syntax (1048334300.py, line 1)